In [27]:
import pandas as pd
# from langchain_gigachat import GigaChat
from langgraph.graph import MessagesState
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import START, StateGraph, END
from langgraph.prebuilt import tools_condition, ToolNode
from IPython.display import Image, display
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from typing import Annotated, List, Tuple, Union, Literal, Dict
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.document_loaders.csv_loader import CSVLoader
import pandas as pd
import re
import subprocess
from tabulate import tabulate

In [28]:
load_dotenv()



True

In [29]:
data = pd.read_csv('scenarios_with_funcs.csv')
data

,scene_name,entry_condition,user_message,category,target,scene_text,funcs
0,Устранение проблем с блокировкой карты,Запрос пользователя связан с одной из перечисл...,"Я не могу использовать свою карту, она, похоже...",Проблемы с картами,1,Идентификация причины блокировки карты. Провер...,"[""check_transaction_history(), confirm_custome..."
1,Подтверждение транзакции,Запрос пользователя связан с одной из перечисл...,Я не вижу на своем счете транзакцию за 5000 ру...,Транзакции,1,Проверка данных транзакции. Верификация источн...,"[""verify_fund_source(), check_transaction_stat..."
2,Проверка статуса страхового полиса,Запрос пользователя связан с одной из перечисл...,Какой статус моего страхового полиса?,Страхование,1,Проверка идентификационной информации клиента....,"[""check_policy_status(), suggest_policy_terms(..."
3,Консультация по инвестиционным стратегиям,Запрос пользователя связан с одной из перечисл...,Какой у вас курс валюты на сегодня?,Инвестиции,0,Анализ финансовых целей клиента. Оценка рисков...,"[""suggest_investment_options()""]"
4,Оформление кредита,Запрос пользователя связан с одной из перечисл...,"Может, расскажете про кредиты? Интересно, что ...",Кредитование,1,Проверка кредитоспособности клиента. Сбор необ...,"[""collect_documents(), suggest_credit_products..."
...,...,...,...,...,...,...,...
190,Проверка активности счета,Запрос пользователя связан с одной из перечисл...,У меня на счету появилось какое-то подозритель...,Безопасность счета,1,Идентификация клиента. Запрос информации о пос...,"[""fetch_recent_transactions()""]"
191,Блокировка банковской карты при утере,Запрос пользователя связан с одной из перечисл...,"Слушай, кажется, я оставил карту в другом горо...",Проблемы с картами,1,Получение информации о состоянии карты. Провер...,"[""verify_client_data(), locate_last_activity()..."
192,Проверка статуса транзакции,Запрос пользователя связан с одной из перечисл...,"Я хочу узнать, прошла ли моя последняя транзак...",Транзакции,1,Получение информации о транзакции. Проверка ст...,"[""check_transaction_status()"", ""analyze_delay_..."
193,Проверка информации о страховом полисе,Запрос пользователя связан с одной из перечисл...,"У меня есть полис на квартиру, хочу узнать, ка...",Страхование,1,Проверка данных клиента. Получение информации ...,"[""fetch_policy_info(), verify_policy_status(),..."


In [30]:
import json
import pandas as pd

def decode_unicode_column(dataset, column_name='funcs'):
    """Декодирует значения в указанной колонке, если они в формате Unicode."""
    decoded_funcs = []
    for funcs in dataset[column_name]:
        if isinstance(funcs, str):
            try:
                decoded = json.loads(funcs)
                decoded_funcs.append(decoded)
            except json.JSONDecodeError:

                decoded_funcs.append(funcs)
        else:
            decoded_funcs.append(funcs)
    dataset[column_name] = decoded_funcs

decode_unicode_column(data, 'funcs')
data['funcs'].head()


0    [check_transaction_history(), confirm_customer...
1    [verify_fund_source(), check_transaction_statu...
2    [check_policy_status(), suggest_policy_terms()...
3                       [suggest_investment_options()]
4     [collect_documents(), suggest_credit_products()]
Name: funcs, dtype: object

In [31]:
def test():
    result = subprocess.run(["python", "test_gen.py"], capture_output=True, text=True)

    if result.returncode != 0:  
        print("Ошибка при выполнении test_gen.py:")
        print(result.stderr) 
        raise RuntimeError("Тестирование завершилось с ошибкой.")  

    print("test_gen.py выполнен успешно:")
    print(result.stdout)
    
    

## Генерация тулзов

In [32]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import json

class Generator:
    def __init__(self, system_prompt_file=''):
        """
        Инициализация генератора с загрузкой системных промптов для разных режимов и API клиента.
        """
        load_dotenv()
        HF_API_KEY = os.getenv('HF_API_KEY')
        api_key = HF_API_KEY
        if api_key is None:
            raise ValueError("API key not found in environment variables.")

        self.client = OpenAI(
            base_url="https://api-inference.huggingface.co/models/Qwen/Qwen2.5-Coder-32B-Instruct/v1/",
            api_key=api_key,
        )

        if system_prompt_file:
            with open(system_prompt_file, "r", encoding="utf-8") as f:
                self.system_prompt = f.read()
        else:
            self.system_prompt = ""

    def prepare_message(self, scene_text, funcs):
        """
        Подготовка сообщения в нужном формате.
        """
        return json.dumps({
            "scen": scene_text,
            "funcs": funcs
        }, ensure_ascii=False)

    def generate(self, user_message, max_tokens=1500):
        """
        Генерирует ответ на основе пользовательского сообщения и системного промпта.
        """
        if not user_message:
            raise ValueError("User message cannot be empty.")

        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_message}
        ]

        try:
            completion = self.client.chat.completions.create(
                model="Qwen2.5-Coder-32B-Instruct",
                messages=messages,
                temperature=0.7,
                n=1,
                max_tokens=max_tokens,
            )

            return completion.choices[0].message.content

        except Exception as e:
            raise RuntimeError(f"Failed to generate response: {e}")

    def process_scenario(self, row):
        """
        Обработка одного сценария из датасета.

        :param row: Строка датафрейма с полями scene_text и funcs
        :return: Сгенерированный код функций
        """
        message = self.prepare_message(row['scene_text'], row['funcs'])
        return self.generate(message)

In [33]:
import random

def get_random_ids(dataset, n=2):
    """Выбирает n случайных ID из датасета."""
    return dataset.sample(n).index.tolist()

def process_generators(generator, data):
    """Process generator with data and create tool implementations."""
    base_import = '''import warnings
warnings.filterwarnings("ignore")
from langchain_gigachat import GigaChat
from langgraph.graph import MessagesState
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import START, StateGraph, END
from langgraph.prebuilt import tools_condition, ToolNode
from langchain_core.tools import tool
from IPython.display import Image, display
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from typing import Annotated, List, Tuple, Union, Literal, Dict
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_community.document_loaders.csv_loader import CSVLoader
import pandas as pd
import re
import ast
from langgraph.prebuilt import create_react_agent

from langchain_community.tools.tavily_search import TavilySearchResults
# Загружаем переменные окружения из .env файла
load_dotenv()

# # Получаем API-ключ из переменной окружения
creds = os.getenv('credentials')
token = os.getenv('OPENAI_API_KEY')

from tabulate import tabulate


# llm = ChatOpenAI(
#     model="gpt-4o-mini", temperature=0.1
# )

llm = GigaChat(
        credentials = creds,
        verify_ssl_certs=False,
        scope = 'GIGACHAT_API_CORP',
        temperature=0.1,
        # model="GigaChat",
        model="GigaChat-Max",
        
        # model="GigaChat-Max-preview",
        # base_url="https://gigachat-preview.devices.sberbank.ru/api/v1/"       
    )

print(f"ТЕСТИРУЕМ АНЕКДОТ ОТ ЛЛМ: {llm.invoke('Расскажи анектод')}")

choose_question_rag_prompt = """
Роль:
Ты — агент, выбирающий наиболее подходящий вопрос к вопросу пользователя.

Задание:
1. Проанализируй предоставленные вопросы и вопрос пользователя.
2. Верни наиболее похожий вопрос, либо ответь что похожего вопроса нет.

Предоставленные вопросы:
{context}

Вопрос пользователя:
{question}

Твоя задача — вернуть из предоставленных вопросов вопрос наиболее похожий на вопрос пользоватeля, либо сообщить об отсутствии такого вопроса (верни 'no'). В ответе должен быть только текст наиболее подходящего вопроса.
Внимание: Удостоверься, что пунктуация возвращаемого тобой вопроса совпадает с той, которая была в момент предоставления вопросов (если в предоставленном вопросе в конце не было знака вопроса, то его добавлять не нужно).
"""

choose_question_rag_template = ChatPromptTemplate.from_template(choose_question_rag_prompt)

question_choose_agent = choose_question_rag_template | llm


def get_embeddings():
    model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
    model_kwargs = {'device': 'cpu'}
    encode_kwargs = {'normalize_embeddings': False}
    embedding = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
        )
    return embedding

def prep_bd():
    file_path = '/home/maksim-litvninov/Documents/'
    vector_store_loaded = FAISS.load_local(folder_path=file_path, embeddings=get_embeddings(), allow_dangerous_deserialization= True)
    loaded_retriever = vector_store_loaded.as_retriever(search_kwargs={'k': 3}, search_type="mmr")
    return loaded_retriever
'''
    base_tools = """
    
    
@tool
def factual_questions(user_question: str) -> str:
    '''Use search_tool when answering factual questions.'''
    search_tool = TavilySearchResults(
        max_results=5,
    )
    return search_tool.invoke(user_question)

@tool
def sberbank_question(bank_question: str) -> str:
    '''Answer questions about Sberbank products and services.'''
    pf = pd.read_csv('FAQ.csv')

    class StateRAG(TypedDict):
        question: str
        context: List[Document]
        answer: str

    def retrieve_question(state: StateRAG):
        retrieved_docs = prep_bd().invoke(state["question"])
        return {"context": retrieved_docs}

    def map_questions_and_answer(state: StateRAG):
        docs_content = "".join(doc.page_content for doc in state["context"])
        question = question_choose_agent.invoke({
            "context": docs_content,
            "question": state["question"]
        })

        if question.content.lower() not in ['no', "'no'"]:
            q = pf.loc[pf['Вопрос'] == question.content[8:], 'Ответ'].values[0]
            return {'answer': q}
        return {"answer": 'No answer available for your question'}

    graph_builder = StateGraph(StateRAG).add_sequence([
        retrieve_question,
        map_questions_and_answer
    ])
    graph_builder.add_edge(START, "retrieve_question")
    graph = graph_builder.compile()

    return f'{graph.invoke({"question": bank_question})["answer"]}'
"""

    tool_collection = '''
def extract_tool_functions(filename):
    """Извлекает названия функций, декорированных @tool, из файла."""
    with open(filename, "r") as file:
        node = ast.parse(file.read(), filename=filename)

    tool_functions = []
    for n in node.body:
        if isinstance(n, ast.FunctionDef):
            for decorator in n.decorator_list:
                if isinstance(decorator, ast.Name) and decorator.id == 'tool':
                    tool_functions.append(n.name)
                    break

    return tool_functions

tool_functions = extract_tool_functions(__file__)

tools = [globals()[name] for name in tool_functions]

assert len(tools) >= 4, "НЕПРАВИЛЬНО СГЕНЕРИРОВАННЫ ФУНКЦИИ"
print("ФУНКЦИИ СГЕНЕРИРОВАНЫ ПРАВИЛЬНО")
print("tools которые мы забиндим в ллмку:", [tool_.name for tool_ in tools])
'''

    base_endings = """
tool_node = ToolNode(tools)
memory = MemorySaver()
"""
    random_ids = get_random_ids(data, 2)
    generated_code_parts = [] 
    print(random_ids)
    for idx in random_ids:
        try:
            row = data.loc[idx]
            generated_code = generator.process_scenario(row)
            generated_code_parts.append(generated_code) 
        except Exception as e:
            print(f"Error in row {idx}: {e}")
            
    full_code = (
    base_import + 
    "\n".join(generated_code_parts) + 
    base_tools + 
    tool_collection + 
    base_endings
)

    return [full_code],random_ids

In [34]:
try:
    generator = Generator(system_prompt_file='system_prompt.txt')
    results,idxs  = process_generators(generator=generator, data=data)
    print(f"Успешно обработано {len(idxs)} сценариев")
except Exception as e:
    print(f"Ошибка при инициализации или выполнении: {e}")

[194, 6]
Успешно обработано 2 сценариев


In [35]:
with open("test_gen.py", "w") as file:
    for line in results:
        cleaned_line = line.replace("\\n", "\n").replace("\\t", "\t")
        file.write(cleaned_line + "\n")



In [36]:
test()

test_gen.py выполнен успешно:
ТЕСТИРУЕМ АНЕКДОТ ОТ ЛЛМ: content='Вот один:\n\n— Доктор, у меня проблема: я всё время забываю, где я живу.\n— А вы не пробовали записывать адрес?\n— Пробовал, но потом забывал, куда положил запись...\n\nНадеюсь, улыбка появилась на твоем лице!' additional_kwargs={} response_metadata={'token_usage': {'prompt_tokens': 18, 'completion_tokens': 59, 'total_tokens': 77}, 'model_name': 'GigaChat-Max:1.0.26.20', 'finish_reason': 'stop'} id='run-37582775-e611-4acf-9390-1d36d7df047d-0'
ФУНКЦИИ СГЕНЕРИРОВАНЫ ПРАВИЛЬНО
tools которые мы забиндим в ллмку: ['set_return_targets', 'suggest_investment_options', 'finalize_investment_strategy', 'analyze_customer_history', 'block_user_account', 'factual_questions', 'sberbank_question']



## Генерация сценариев

In [37]:
slovar = {
    'Название сценария': [
        data.loc[idxs[0]]['scene_name'], 
        data.loc[idxs[1]]['scene_name']
    ],
    'Условия входа': [
        data.loc[idxs[0]]['entry_condition'], 
        data.loc[idxs[1]]['entry_condition']
    ],
    'Описание сценария': [
        data.loc[idxs[0]]['scene_text'], 
        data.loc[idxs[1]]['scene_text']
    ],
    'Справочная информация': [
        f"""
        Справочной информации нет для текущего сценария, используйте функции {", ".join(data.loc[idxs[0]]['funcs'])}.
        """,
        f"""
        Справочной информации нет для текущего сценария, используйте функции {", ".join(data.loc[idxs[1]]['funcs'])}.
        """
    ]
}

with open("test_gen.py", "a", encoding="utf-8") as file:
    file.write("slovar = {\n")
    for key, values in slovar.items():
        file.write(f"    '{key}': [\n")
        for value in values:
            value = str(value).replace("'''", "\\'\\'\\'")
            file.write(f"        '''{value}''',\n")
        file.write("    ],\n")
    file.write("}\n\n")
    file.write("df = pd.DataFrame(slovar)\n")
    
    file.write("assert df.shape == (2, 4), 'ОШИБКА ГЕНЕРАЦИИ ДАТАФРЕЙМА СЦЕНАРИЕВ'\n")
    file.write("print('СЦЕНАРИИ СОЗДАНЫ ПРАВИЛЬНО')\n")
    file.write("print(tabulate(df, headers='keys', tablefmt='psql'))\n")


In [38]:
test()

test_gen.py выполнен успешно:
ТЕСТИРУЕМ АНЕКДОТ ОТ ЛЛМ: content='Вот один:\n\n— Доктор, у меня проблема: я всё время забываю, где я живу.\n— А вы не пробовали записывать адрес?\n— Пробовал, но потом забывал, куда положил запись...\n\nНадеюсь, улыбка появилась на твоем лице!' additional_kwargs={} response_metadata={'token_usage': {'prompt_tokens': 18, 'completion_tokens': 59, 'total_tokens': 77}, 'model_name': 'GigaChat-Max:1.0.26.20', 'finish_reason': 'stop'} id='run-6f006675-be9b-49b8-bb2a-c7aa1b6a4ce9-0'
ФУНКЦИИ СГЕНЕРИРОВАНЫ ПРАВИЛЬНО
tools которые мы забиндим в ллмку: ['set_return_targets', 'suggest_investment_options', 'finalize_investment_strategy', 'analyze_customer_history', 'block_user_account', 'factual_questions', 'sberbank_question']
СЦЕНАРИИ СОЗДАНЫ ПРАВИЛЬНО
+----+-------------------------------------+----------------------------------------------------------+------------------------------------------------------------------------------------------------------------------

## Заполнение структуры агента

### Пожалуйста вставьте в ячейку ниже ПУТЬ к файлу, где будет вся логика агента ПОСЛЕ ТУЛЗОВ. в случае с ноутбуком rebuild_giga.ipynb ВСЁ от "планировщики сценарный и динамический (чисто на функциях)" ДО "Тест по сообщению". 

#### ИМПОРТЫ С НАЧАЛА НОУТБУКА ВСТАВЛЯТЬ НЕ НАДО ОНИ САМИ ПОДТЯНУТСЯ

In [39]:
"""ВМЕСТО base_agent.py - УКАЖИТЕ ПУТЬ К СВОЕМУ ФАЙЛУ."""
with open("base_agent.py", "r", encoding="utf-8") as source_file, open("test_gen.py", "a", encoding="utf-8") as target_file:
    for line in source_file:
        target_file.write(line)

with open("test_gen.py", "a", encoding="utf-8") as file:
    file.write("\n")
    file.write("print('НОЛЬ ОШИБОК')\n")

In [40]:
test()

test_gen.py выполнен успешно:
ТЕСТИРУЕМ АНЕКДОТ ОТ ЛЛМ: content='Вот один:\n\n— Доктор, у меня проблема: я всё время забываю, где я живу.\n— А вы не пробовали записывать адрес?\n— Пробовал, но потом забывал, куда положил запись...\n\nНадеюсь, улыбка появилась на твоем лице!' additional_kwargs={} response_metadata={'token_usage': {'prompt_tokens': 18, 'completion_tokens': 59, 'total_tokens': 77}, 'model_name': 'GigaChat-Max:1.0.26.20', 'finish_reason': 'stop'} id='run-51f7fdf8-db45-4fa2-9456-ea4734e64d15-0'
ФУНКЦИИ СГЕНЕРИРОВАНЫ ПРАВИЛЬНО
tools которые мы забиндим в ллмку: ['set_return_targets', 'suggest_investment_options', 'finalize_investment_strategy', 'analyze_customer_history', 'block_user_account', 'factual_questions', 'sberbank_question']
СЦЕНАРИИ СОЗДАНЫ ПРАВИЛЬНО
+----+-------------------------------------+----------------------------------------------------------+------------------------------------------------------------------------------------------------------------------

## Мок диалог, для проверки работоспособности.

In [41]:
with open("test_gen.py", "a", encoding="utf-8") as target_file:
    target_file.write("""\
import pandas as pd
import uuid

# Определение тестовой структуры
CACHE_DF = pd.DataFrame(columns=["plan_id", "plan_text", "plan_status", "completed_items", "start_mes", "summary"])
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 10}
_printed = set()

testing_dialog = [
    "Привет, как дела?",
    "Супер, меня зовут Максим",
    "Как меня зовут",
    "Расскажи про кошек",
    "Теперь про собак",
]

# Проверка диалогов через stream
for question in testing_dialog:
    events = app.stream(
        {"messages": ("user", question)}, config, stream_mode="values"
    )
    for event in events:
        _print_event(event, _printed)
""")


In [42]:
test()

Ошибка при выполнении test_gen.py:
Traceback (most recent call last):
  File "/home/maksim-litvninov/Documents/planner/test_gen.py", line 899, in <module>
    for event in events:
                 ^^^^^^
  File "/home/maksim-litvninov/miniconda3/envs/work/lib/python3.13/site-packages/langgraph/pregel/__init__.py", line 1670, in stream
    for _ in runner.tick(
             ~~~~~~~~~~~^
        loop.tasks.values(),
        ^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
        get_waiter=get_waiter,
        ^^^^^^^^^^^^^^^^^^^^^^
    ):
    ^
  File "/home/maksim-litvninov/miniconda3/envs/work/lib/python3.13/site-packages/langgraph/pregel/runner.py", line 171, in tick
    run_with_retry(
    ~~~~~~~~~~~~~~^
        t,
        ^^
    ...<4 lines>...
        },
        ^^
    )
    ^
  File "/home/maksim-litvninov/miniconda3/envs/work/lib/python3.13/site-packages/langgraph/pregel/retry.py", line 40, in run_with_retry
    return task.proc.invoke(task.input, config)
           ~~~~~~~~~~~~~~~~^^^

RuntimeError: Тестирование завершилось с ошибкой.